In [2]:
from pathlib import Path
import os

cwd = Path.cwd()
project_root = cwd.parents[0]

DATASET_NAME = "kiwi"
SPLIT_TYPE = "acrossyear"
SEQUENCE = "ordered"
EXPERIMENT_TAG = f"birdnet_lstm_{SEQUENCE}"
RUN_NAME = f"{DATASET_NAME}_{SPLIT_TYPE}_{EXPERIMENT_TAG}"

log_root = project_root / "logs"

# Search all best checkpoints.
ckpt_candidates = sorted(
    log_root.glob(f"**/{RUN_NAME}/version_*/checkpoints/best-*.ckpt"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

print(f"Found {len(ckpt_candidates)} candidate checkpoints.")

for i, p in enumerate(ckpt_candidates[:10]):
    print(i, p)

if not ckpt_candidates:
    raise FileNotFoundError(
        f"No checkpoint found for RUN_NAME={RUN_NAME}. "
        "Check LOG_ROOT, DATASET_NAME, SPLIT_TYPE, and SEQUENCE."
    )

checkpoint_path = ckpt_candidates[0]
print("Selected checkpoint:", checkpoint_path)


Found 17 candidate checkpoints.
0 /teamspace/studios/this_studio/logs/kiwi/acrossyear/kiwi_acrossyear_birdnet_lstm_ordered/version_32/checkpoints/best-kiwi-acrossyear-birdnet_lstm_ordered-epoch=11-val_f1=0.9678.ckpt
1 /teamspace/studios/this_studio/logs/kiwi/acrossyear/kiwi_acrossyear_birdnet_lstm_ordered/version_30/checkpoints/best-kiwi-acrossyear-RNN-birdnet_lstm_ordered-epoch=11-val_f1=0.9678.ckpt
2 /teamspace/studios/this_studio/logs/kiwi/acrossyear/kiwi_acrossyear_birdnet_lstm_ordered/version_28/checkpoints/best-kiwi-acrossyear-RNN-birdnet_lstm_ordered-epoch=15-val_f1=0.9918.ckpt
3 /teamspace/studios/this_studio/logs/kiwi/acrossyear/kiwi_acrossyear_birdnet_lstm_ordered/version_26/checkpoints/best-kiwi-acrossyear-RNN-birdnet_lstm_ordered-epoch=15-val_f1=0.9119.ckpt
4 /teamspace/studios/this_studio/logs/kiwi/acrossyear/kiwi_acrossyear_birdnet_lstm_ordered/version_24/checkpoints/best-kiwi-acrossyear-RNN-birdnet_lstm_ordered-epoch=16-val_f1=0.9711.ckpt
5 /teamspace/studios/this_studio

In [3]:
import torch
from torch import nn
from lightning import LightningModule
import torchmetrics
import logging
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)


class BirdsongClassifier(LightningModule):
    def __init__(
        self,
        embedding_dim: int,
        hidden_dim: int,
        num_classes: int,
        lr: float = 1e-3,
        bidirectional: bool = False,
        dropout: float = 0.0,
        use_mean_pool: bool = False,
        class_weights: torch.Tensor | None = None,
    ):
        super().__init__()
        self.save_hyperparameters(ignore=["class_weights"])
        self.class_weights = class_weights

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if 1 < 1 else 0.0,
        )

        out_dim = hidden_dim * (2 if bidirectional else 1)

        self.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(out_dim, num_classes),
        )

        self.train_accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)
        self.val_accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)
        self.test_accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)
        self.val_f1 = torchmetrics.F1Score(task="multiclass", num_classes=num_classes, average="macro")
        self.test_f1 = torchmetrics.F1Score(task="multiclass", num_classes=num_classes, average="macro")

        self.criterion = nn.CrossEntropyLoss(weight=None)
        self.use_mean_pool = use_mean_pool

    def setup(self, stage=None):
        if self.class_weights is not None:
            self.criterion = nn.CrossEntropyLoss(weight=self.class_weights.to(self.device))

    def forward(self, x: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        packed = pack_padded_sequence(
            x,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )

        packed_out, (h_n, c_n) = self.lstm(packed)

        if self.use_mean_pool:
            out, lens = pad_packed_sequence(packed_out, batch_first=True)
            lens = lens.to(out.device).clamp_min(1).unsqueeze(1)
            feats = out.sum(dim=1) / lens
        else:
            feats = h_n[-1]

        logits = self.fc(feats)
        return logits

    def training_step(self, batch, batch_idx):
        x, lengths, y = batch
        logits = self(x, lengths)
        loss = self.criterion(logits, y)
        acc = self.train_accuracy(logits, y)

        self.log("train_loss", loss, prog_bar=True)
        self.log("train_acc", acc, prog_bar=True)

        logger.info(
            f"Batch {batch_idx} - Train Loss: {loss.item():.4f} - Train Acc: {acc:.4f}"
        )

        return loss

    def validation_step(self, batch, batch_idx):
        x, lengths, y = batch
        logits = self(x, lengths)
        loss = self.criterion(logits, y)
        acc = self.val_accuracy(logits, y)
        f1 = self.val_f1(logits, y)

        self.log("val_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log("val_acc", acc, prog_bar=True, on_step=False, on_epoch=True)
        self.log("val_f1", f1, prog_bar=True, on_step=False, on_epoch=True)

    def test_step(self, batch, batch_idx):
        x, lengths, y = batch
        logits = self(x, lengths)
        loss = self.criterion(logits, y)
        acc = self.test_accuracy(logits, y)
        f1 = self.test_f1(logits, y)

        self.log("test_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log("test_acc", acc, prog_bar=True, on_step=False, on_epoch=True)
        self.log("test_f1", f1, prog_bar=True, on_step=False, on_epoch=True)

        logger.info(
            f"Batch {batch_idx} - Test Loss: {loss.item():.4f} - Test Acc: {acc:.4f}"
        )

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)


print("BirdsongClassifier class defined.")

BirdsongClassifier class defined.


In [4]:
ckpt = torch.load(str(checkpoint_path), map_location="cpu")

print("Checkpoint keys:")
print(ckpt.keys())

print("\nHyperparameters:")
for key, value in ckpt.get("hyper_parameters", {}).items():
    print(f"{key}: {value}")

print("\nFirst state_dict keys:")
for key in list(ckpt["state_dict"].keys())[:20]:
    print(key)

Checkpoint keys:
dict_keys(['epoch', 'global_step', 'pytorch-lightning_version', 'state_dict', 'loops', 'callbacks', 'optimizer_states', 'lr_schedulers', 'hparams_name', 'hyper_parameters'])

Hyperparameters:
embedding_dim: 1024
hidden_dim: 256
num_classes: 20
lr: 0.001
bidirectional: False
dropout: 0.0
use_mean_pool: False

First state_dict keys:
lstm.weight_ih_l0
lstm.weight_hh_l0
lstm.bias_ih_l0
lstm.bias_hh_l0
fc.1.weight
fc.1.bias


In [5]:
device = torch.device("cpu")

model = BirdsongClassifier.load_from_checkpoint(
    str(checkpoint_path),
    map_location=device,
)

model.eval()
model.to(device)

print("Model loaded successfully.")
print("Model device:", next(model.parameters()).device)
print("Model mode:", "training" if model.training else "evaluation")

Model loaded successfully.
Model device: cpu
Model mode: evaluation


In [6]:
import pandas as pd
import numpy as np
import torch

embedding_parquet_path = (
    project_root
    / "runtime_benchmark"
    / "benchmark_kiwi_30s_birdnet_embeddings.parquet"
)

embedding_df = pd.read_parquet(embedding_parquet_path)

print("Loaded Parquet:", embedding_parquet_path)
print("DataFrame shape:", embedding_df.shape)
print("Columns preview:")
print(embedding_df.columns[:12].tolist())

Loaded Parquet: /teamspace/studios/this_studio/runtime_benchmark/benchmark_kiwi_30s_birdnet_embeddings.parquet
DataFrame shape: (10, 1031)
Columns preview:
['file_name', 'original_audio_path', 'window_index', 'start_time', 'end_time', 'padded_duration_s', 'n_birdnet_windows', 'emb_0000', 'emb_0001', 'emb_0002', 'emb_0003', 'emb_0004']


In [7]:
embedding_cols = sorted([
    col for col in embedding_df.columns
    if col.startswith("emb_")
])

embedding_df = embedding_df.sort_values("window_index").reset_index(drop=True)

embeddings = embedding_df[embedding_cols].to_numpy(dtype=np.float32)

x = torch.tensor(embeddings, dtype=torch.float32).unsqueeze(0).to(device)
lengths = torch.tensor([embeddings.shape[0]], dtype=torch.long).to(device)

print("Number of embedding columns:", len(embedding_cols))
print("Embedding matrix shape:", embeddings.shape)
print("Input tensor shape:", x.shape)
print("Lengths:", lengths)

Number of embedding columns: 1024
Embedding matrix shape: (10, 1024)
Input tensor shape: torch.Size([1, 10, 1024])
Lengths: tensor([10])


In [8]:
with torch.inference_mode():
    logits = model(x, lengths)
    pred = torch.argmax(logits, dim=1)

print("Logits shape:", logits.shape)
print("Predicted class index:", pred.item())

Logits shape: torch.Size([1, 20])
Predicted class index: 15


In [9]:
import time
import gc
import pandas as pd

N_REPEATS = 1000

runtime_csv_path = (
    project_root
    / "runtime_benchmark"
    / "runtime_lstm_inference.csv"
)

# Warm-up run. This avoids including first-call overhead in the benchmark.
with torch.inference_mode():
    _ = model(x, lengths)

rows = []

for i in range(N_REPEATS):
    t0 = time.perf_counter()

    with torch.inference_mode():
        logits = model(x, lengths)
        pred = torch.argmax(logits, dim=1)

    t1 = time.perf_counter()

    rows.append({
        "repeat": i + 1,
        "n_birdnet_windows": int(embeddings.shape[0]),
        "embedding_dim": int(embeddings.shape[1]),
        "num_classes": int(logits.shape[1]),
        "lstm_inference_s": t1 - t0,
        "predicted_class_index": int(pred.item()),
    })

runtime_lstm_df = pd.DataFrame(rows)
runtime_lstm_df.to_csv(runtime_csv_path, index=False)

print(runtime_lstm_df.describe().T[["mean", "std"]])
print("Saved runtime CSV to:", runtime_csv_path)

                              mean         std
repeat                  500.500000  288.819436
n_birdnet_windows        10.000000    0.000000
embedding_dim          1024.000000    0.000000
num_classes              20.000000    0.000000
lstm_inference_s          0.000866    0.000470
predicted_class_index    15.000000    0.000000
Saved runtime CSV to: /teamspace/studios/this_studio/runtime_benchmark/runtime_lstm_inference.csv


In [10]:
import numpy as np
import pandas as pd

birdnet_runtime_path = (
    project_root
    / "runtime_benchmark"
    / "runtime_birdnet_padding_embedding.csv"
)

lstm_runtime_path = (
    project_root
    / "runtime_benchmark"
    / "runtime_lstm_inference.csv"
)

summary_csv_path = (
    project_root
    / "runtime_benchmark"
    / "runtime_summary_cpu_postsegmentation.csv"
)

birdnet_runtime = pd.read_csv(birdnet_runtime_path)
lstm_runtime = pd.read_csv(lstm_runtime_path)

padding_mean = birdnet_runtime["padding_and_temp_write_s"].mean()
padding_std = birdnet_runtime["padding_and_temp_write_s"].std()

birdnet_mean = birdnet_runtime["birdnet_embedding_extraction_s"].mean()
birdnet_std = birdnet_runtime["birdnet_embedding_extraction_s"].std()

birdnet_total_mean = birdnet_runtime["total_padding_plus_embedding_s"].mean()
birdnet_total_std = birdnet_runtime["total_padding_plus_embedding_s"].std()

lstm_mean = lstm_runtime["lstm_inference_s"].mean()
lstm_std = lstm_runtime["lstm_inference_s"].std()

total_pipeline_mean = padding_mean + birdnet_mean + lstm_mean
total_pipeline_std = np.sqrt(
    padding_std**2 + birdnet_std**2 + lstm_std**2
)

total_pipeline_alt_mean = birdnet_total_mean + lstm_mean
total_pipeline_alt_std = np.sqrt(
    birdnet_total_std**2 + lstm_std**2
)

summary_df = pd.DataFrame({
    "stage": [
        "audio_loading_padding_temp_write",
        "birdnet_embedding_extraction",
        "lstm_classification",
        "total_postsegmentation_pipeline_components",
        "total_postsegmentation_pipeline_from_birdnet_total",
    ],
    "mean_s": [
        padding_mean,
        birdnet_mean,
        lstm_mean,
        total_pipeline_mean,
        total_pipeline_alt_mean,
    ],
    "std_s": [
        padding_std,
        birdnet_std,
        lstm_std,
        total_pipeline_std,
        total_pipeline_alt_std,
    ],
    "mean_ms": [
        padding_mean * 1000,
        birdnet_mean * 1000,
        lstm_mean * 1000,
        total_pipeline_mean * 1000,
        total_pipeline_alt_mean * 1000,
    ],
    "std_ms": [
        padding_std * 1000,
        birdnet_std * 1000,
        lstm_std * 1000,
        total_pipeline_std * 1000,
        total_pipeline_alt_std * 1000,
    ],
})

summary_df.to_csv(summary_csv_path, index=False)

print(summary_df)
print("Saved summary CSV to:", summary_csv_path)

                                               stage    mean_s     std_s  \
0                   audio_loading_padding_temp_write  0.010014  0.000658   
1                       birdnet_embedding_extraction  0.807770  0.015932   
2                                lstm_classification  0.000866  0.000470   
3         total_postsegmentation_pipeline_components  0.818649  0.015952   
4  total_postsegmentation_pipeline_from_birdnet_t...  0.818652  0.016015   

      mean_ms     std_ms  
0   10.014102   0.657766  
1  807.769609  15.931582  
2    0.865593   0.470044  
3  818.649304  15.952082  
4  818.652476  16.015050  
Saved summary CSV to: /teamspace/studios/this_studio/runtime_benchmark/runtime_summary_cpu_postsegmentation.csv
